In [3]:
!pip install -q -U google-genai sentence-transformer chromadb langchain-text-splitters py pdf

ERROR: Could not find a version that satisfies the requirement sentence-transformer (from versions: none)
ERROR: No matching distribution found for sentence-transformer


In [4]:
!pip install chromadb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 34.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 31.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.1/23.1 MB 36.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 2.0 MB/s eta 0:00:00
  Attempting uninstall: opentelemetry-api
    Found exi

In [5]:
!pip install langchain-text-splitters

In [6]:
!pip install pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 393.8/393.8 kB 4.8 MB/s eta 0:00:00


In [7]:
import os
from google.colab import userdata, files
from google import genai
from sentence_transformers import SentenceTransformer
import chromadb
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pypdf import PdfReader

In [8]:
api_key = os.environ.get("GOOGLE_API_KEY") or os.environ.get("G_A_K")

if not api_key:
    try:
        api_key = userdata.get("Swaraj_Kharpude")
    except Exception:
        pass

if not api_key:
    raise ValueError("Google API key not found.")

os.environ["GOOGLE_API_KEY"] = api_key

client = genai.Client(api_key=api_key)

In [9]:
print(" Please upload one or more PDF files:")
uploaded = files.upload()

pdf_texts = []
for filename in uploaded.keys():
  if filename.endswith('.pdf'):
    reader = PdfReader(filename)
    text = ""
    for page_num, page in enumerate(reader.pages):
      page_text = page.extract_text()
      if page_text:
        text += f"\n --- Page {page_num +1} ---\n" + page_text
    pdf_texts.append(text)
    print(f" Loaded '{filename}' ({len(reader.pages)} pages).")

if not pdf_texts:
  raise ValueError("No valid PDF files uploaded. Please re-run and upload a .pdf")

full_pdf_content = "\n\n".join(pdf_texts)

 Please upload one or more PDF files:


Saving Practical No 4_105.pdf to Practical No 4_105.pdf
 Loaded 'Practical No 4_105.pdf' (6 pages).


In [10]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)
chunks = text_splitter.split_text(full_pdf_content)
print(f" Extracted and split document into {len(chunks)} text chunks.")

 Extracted and split document into 3 text chunks.


In [11]:
print(" Loading embedding model and building vector index...")
embedder = SentenceTransformer("all-MiniLM-L6-v2")
chroma_client = chromadb.Client()

#Reset collection for clean execution
try:
  chroma_client.delete_collection(name="pdf_rag_collection")
except Exception:
  pass

collection = chroma_client.create_collection(name="pdf_rag_collection")

#Embed chunks in batches
chunk_embeddings = embedder.encode(chunks).tolist()
chunks_ids = [f"doc_chunks_{i}" for i in range(len(chunks))]

collection.add(
    documents=chunks,
    embeddings=chunk_embeddings,
    ids=chunks_ids
)

print("PDF Vector Indexing Complete\n")

 Loading embedding model and building vector index...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

PDF Vector Indexing Complete



In [14]:
def retrieve_pdf_context(query: str, top_k:int=3) -> list[str]:
  query_embedding = embedder.encode(query).tolist()
  results = collection.query(
      query_embeddings=query_embedding,
      n_results=top_k
  )
  return results["documents"][0]

def ask_pdf(query: str):
    context_passages = retrieve_pdf_context(query, top_k=3)
    context_str = "\n".join(f"- {p}" for p in context_passages)

    prompt = f"""
You are an intelligent document analysis assistant.

Answer the question using ONLY the information provided in the PDF context.

If the information is not contained within the provided context, state clearly:
"I cannot find the answer in the provided PDF."

PDF Context:
{context_str}

Question: {query}

Answer:
"""

    response = client.models.generate_content(
        model="gemini-3.5-flash",
        contents=prompt
    )

    return response.text, context_passages



In [ ]:
print("=" * 60)
print(" PDF CHATBOT READY! Type your question below (or type 'exit' to quit).")
print("=" * 60)

while True:
  user_query = input("\nAsk a question about your PDF: ")
  if user_query.lower() in ["exit","quit","q"]:
    print(" Exiting PDF Chatbot. Goodbye!")
    break
  if not user_query.strip():
    continue

  answer, context = ask_pdf(user_query)

  print("\n--- RETRIEVED PDF SNIPPETS ---")
  for i,snippet in enumerate(context, 1):
    print(f"[{i}] {snippet[:150]}...")

  print("\n--- GEMINI RESPONSE ---")
  print(answer)
  print("=" * 60)


 PDF CHATBOT READY! Type your question below (or type 'exit' to quit).

Ask a question about your PDF: Tell me what this pdf is about give brief summary about it

--- RETRIEVED PDF SNIPPETS ---
[1] --- Page 4 ---
Tejaswini Patil_105 
describe formatted student; 
 
student order by id desc; 
 
 
 
 
 
 

 --- Page 5 ---
Tejaswini Patil_105 
 
sele...
[2] --- Page 2 ---
Tejaswini Patil_105 
 
 
 
 
 
 
 
 
 
 

 --- Page 3 ---
Tejaswini Patil_105 
create database demo; 
use demo; 
create table student( ...
[3] --- Page 1 ---
Tejaswini Patil_105 
Practical No:4 
Aim: Basic Hive Commands 
 
 
 

 --- Page 2 ---
Tejaswini Patil_105...

--- GEMINI RESPONSE ---
Based on the provided PDF context, this document is a report or record for **Practical No: 4**, authored by **Tejaswini Patil_105**. 

The aim of the practical is **Basic Hive Commands**. The document provides a step-by-step demonstration of these commands, which include:
* Creating and using a database (`demo`).
* Creating a table (`s